# Merge Attached PawTrainer Project Files
This notebook inspects the current workspace, compares it against the attached PawTrainer source folders, and copies the source files into the current Laravel project folder.

## 1. Inspect Current Project Structure
The following cell lists all current files and folders in the target project workspace.

In [ ]:
import os
from pathlib import Path
root = Path(r'c:\Semester 4\TIS\Proyek_Akhir\pawtrainer')
for dirpath, dirnames, filenames in os.walk(root):
    rel = Path(dirpath).relative_to(root)
    print(rel if rel != Path('.') else '.')
    for d in sorted(dirnames):
        print('  D', d)
    for f in sorted(filenames):
        print('  F', f)
    if rel.parts and rel.parts[0] == 'vendor':
        # skip deep vendor listing to keep output manageable
        dirnames[:] = []


## 2. Inspect Source Project Files
This cell lists the contents of the attached source folders available under Downloads.

In [ ]:
from pathlib import Path
source_roots = [
    Path(r'C:\Users\LENOVO\Downloads\PawTrainer_Backend_v2\PawTrainer'),
    Path(r'C:\Users\LENOVO\Downloads\PawTrainer_Backend\PawTrainer'),
    Path(r'C:\Users\LENOVO\Downloads\files')
]
for root_path in source_roots:
    print('SOURCE:', root_path)
    if not root_path.exists():
        print('  MISSING', root_path)
        continue
    for dirpath, dirnames, filenames in os.walk(root_path):
        rel = Path(dirpath).relative_to(root_path)
        print(rel if rel != Path('.') else '.')
        for d in sorted(dirnames):
            print('  D', d)
        for f in sorted(filenames):
            print('  F', f)
        if rel.parts and rel.parts[0] == 'PawTrainer_Final':
            dirnames[:] = []
    print('\n')


## 3. Copy Files into Target Project
This cell copies the backend source files from the attached `PawTrainer_Backend_v2` folder into the current project.


In [ ]:
import shutil
from pathlib import Path

source_root = Path(r'C:\Users\LENOVO\Downloads\PawTrainer_Backend_v2\PawTrainer')
target_root = Path(r'c:\Semester 4\TIS\Proyek_Akhir\pawtrainer')

# Files and folders to merge into the current project
items_to_copy = [
    'app',
    'config',
    'database',
    'routes',
    '.env.example',
    'README.md',
    'API_RESPONSES.md',
    'PawTrainer.postman_collection.json',
    'PawTrainer_v2.postman_collection.json'
]

for item in items_to_copy:
    src = source_root / item
    dest = target_root / item
    if not src.exists():
        print('SKIP missing source:', src)
        continue
    if src.is_dir():
        print('Copying directory:', src, '->', dest)
        if not dest.exists():
            dest.mkdir(parents=True, exist_ok=True)
        for root_path, dirs, files in os.walk(src):
            rel = Path(root_path).relative_to(src)
            current_target = dest / rel
            current_target.mkdir(parents=True, exist_ok=True)
            for file_name in files:
                source_file = Path(root_path) / file_name
                dest_file = current_target / file_name
                shutil.copy2(source_file, dest_file)
                print('  Wrote:', dest_file)
    else:
        print('Copying file:', src, '->', dest)
        shutil.copy2(src, dest)
        print('  Wrote:', dest)

# Optional: copy frontend static files into public/frontend
frontend_source = Path(r'C:\Users\LENOVO\Downloads\files')
frontend_dest = target_root / 'public' / 'frontend'
if frontend_source.exists():
    frontend_dest.mkdir(parents=True, exist_ok=True)
    for file_name in os.listdir(frontend_source):
        source_file = frontend_source / file_name
        if source_file.is_file():
            dest_file = frontend_dest / file_name
            shutil.copy2(source_file, dest_file)
            print('Copied frontend file:', dest_file)


## 4. Resolve Directory and File Conflicts
If there are conflicts from existing files, this cell lists those files and keeps the merged source version.


In [ ]:
from pathlib import Path

conflicts = []
source_dirs = [
    Path(r'C:\Users\LENOVO\Downloads\PawTrainer_Backend_v2\PawTrainer\app'),
    Path(r'C:\Users\LENOVO\Downloads\PawTrainer_Backend_v2\PawTrainer\config'),
    Path(r'C:\Users\LENOVO\Downloads\PawTrainer_Backend_v2\PawTrainer\database'),
    Path(r'C:\Users\LENOVO\Downloads\PawTrainer_Backend_v2\PawTrainer\routes')
]
target_root = Path(r'c:\Semester 4\TIS\Proyek_Akhir\pawtrainer')

for source_dir in source_dirs:
    if source_dir.exists():
        for root_path, dirs, files in os.walk(source_dir):
            rel = Path(root_path).relative_to(source_dir)
            for file_name in files:
                target_file = target_root / source_dir.name / rel / file_name
                if target_file.exists():
                    conflicts.append(str(target_file))

if conflicts:
    print('Found existing files that were overwritten from source:')
    for c in conflicts:
        print('  ', c)
else:
    print('No conflicts detected or no existing files were overwritten.')


## 5. Verify Project Structure and Run Basic Checks
This cell validates the merged file placement and checks the critical backend files are present.


In [ ]:
from pathlib import Path

target_root = Path(r'c:\Semester 4\TIS\Proyek_Akhir\pawtrainer')
expected = [
    target_root / 'app' / 'Models' / 'User.php',
    target_root / 'app' / 'Http' / 'Controllers' / 'Api' / 'AuthController.php',
    target_root / 'database' / 'migrations' / '2024_01_01_000001_create_users_table.php',
    target_root / 'routes' / 'api.php',
    target_root / '.env.example'
]
for path in expected:
    print(path, 'EXISTS' if path.exists() else 'MISSING')

print('\nTop-level merged files:')
for path in ['README.md', 'API_RESPONSES.md', 'PawTrainer.postman_collection.json', 'PawTrainer_v2.postman_collection.json']:
    print(path, 'EXISTS' if (target_root / path).exists() else 'MISSING')

print('\nStatic frontend files copied to public/frontend:')
frontend_dir = target_root / 'public' / 'frontend'
if frontend_dir.exists():
    for file in sorted(frontend_dir.iterdir()):
        print('  ', file.name)
else:
    print('  public/frontend not found')
